# 03b — Student Model Training
**Requires 03a to have run first.**

Session budget on Kaggle:
- `beam_M1` (10k pairs): **~45 min** on T4  
- `beam_M10` (100k pairs): **~6 h** on T4  

This notebook:
1. Loads cached tokenized data from disk (no re-tokenizing)
2. Builds the student Transformer
3. Trains with early stopping, saves best checkpoint
4. Can be **re-run safely** — resumes from latest checkpoint if one exists

### Output
- `notebooks/models/<run_name>_best.pt` — best model checkpoint
- `results/training_log_<run_name>.csv` — per-epoch metrics

In [ ]:
# !pip install --quiet torch sentencepiece tqdm pandas

In [ ]:
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

import sentencepiece as spm
import sacrebleu

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────
ROOT       = Path("..")
MODEL_DIR  = ROOT / "notebooks" / "models"
CACHE_DIR  = MODEL_DIR / "cache"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

VOCAB_INFO_PATH = MODEL_DIR / "vocab_info.json"
assert VOCAB_INFO_PATH.exists(), "❌ Run 03a first — vocab_info.json not found"

with open(VOCAB_INFO_PATH) as f:
    vi = json.load(f)

VOCAB_SIZE = vi["vocab_size"]
PAD_ID     = vi["pad_id"]
BOS_ID     = vi["bos_id"]
EOS_ID     = vi["eos_id"]
MAX_LENGTH = vi["max_length"]
SPM_MODEL  = vi["spm_model"]

sp = spm.SentencePieceProcessor(model_file=SPM_MODEL)

print(f"✓ Vocab: {VOCAB_SIZE} | PAD={PAD_ID} BOS={BOS_ID} EOS={EOS_ID} | MaxLen={MAX_LENGTH}")

In [ ]:
# ── TRAINING CONFIG ───────────────────────────────────────────────────────
# Change DATASET to train a different variant.
# Start with beam_M1 (~45 min). Move to beam_M10 (~6 h) once pipeline is verified.

DATASET      = "beam_M1"   # beam_M1 | beam_M10 | top_p_M10 | top_k_M10 | dbs_M10 | mbr_M10
MODEL_SIZE   = "A"         # A = 65M (paper), B = 5M (debug/fast)

# Hyperparameters
BATCH_SIZE       = 32
GRADIENT_ACCUM   = 2       # effective batch = 64
LEARNING_RATE    = 1e-3
WARMUP_STEPS     = 4000
MAX_EPOCHS       = 50
EARLY_STOP       = 5       # stop if no dev improvement for N epochs
LABEL_SMOOTHING  = 0.1
WEIGHT_DECAY     = 1e-4
CLIP_GRAD        = 1.0
USE_FP16         = torch.cuda.is_available()
NUM_WORKERS      = 2

# Model architectures
MODEL_CFGS = {
    "A": dict(d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, dropout=0.3),
    "B": dict(d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=512,  dropout=0.1),
}
model_cfg = MODEL_CFGS[MODEL_SIZE]

RUN_NAME = f"student_{DATASET}_opt{MODEL_SIZE}"
CKPT_BEST   = MODEL_DIR / f"{RUN_NAME}_best.pt"
CKPT_LATEST = MODEL_DIR / f"{RUN_NAME}_latest.pt"
LOG_CSV     = RESULTS_DIR / f"training_log_{RUN_NAME}.csv"

print(f"Run: {RUN_NAME}")
print(f"Dataset cache: {CACHE_DIR / f'cache_{DATASET}.pt'}")
assert (CACHE_DIR / f"cache_{DATASET}.pt").exists(), f"❌ Cache not found. Run 03a first."

In [ ]:
# ── DATA LOADING ──────────────────────────────────────────────────────────

class CachedDataset(Dataset):
    """Wraps the pre-tokenized list-of-dicts saved by 03a."""
    def __init__(self, data: List[Dict]):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    src_padded = pad_sequence([ex["src"] for ex in batch], batch_first=True, padding_value=PAD_ID)
    tgt_padded = pad_sequence([ex["tgt"] for ex in batch], batch_first=True, padding_value=PAD_ID)
    return {"src": src_padded, "tgt": tgt_padded}


print(f"Loading cache: cache_{DATASET}.pt ...")
raw_data = torch.load(CACHE_DIR / f"cache_{DATASET}.pt", weights_only=False)
print(f"  {len(raw_data)} pairs loaded")

# 95/5 train/val split
val_size   = max(200, int(0.05 * len(raw_data)))
train_size = len(raw_data) - val_size
gen = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(CachedDataset(raw_data), [train_size, val_size], generator=gen)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

print(f"  Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"  Train batches/epoch: {len(train_loader)}")

# FLORES dev (for per-epoch BLEU check)
flores_dev_raw = json.load(open(CACHE_DIR / "raw_flores_dev.json", encoding="utf-8"))
FLORES_DEV_SRC = flores_dev_raw["src"]
FLORES_DEV_REF = flores_dev_raw["ref"]
print(f"  FLORES dev: {len(FLORES_DEV_SRC)} sentences")

In [ ]:
# ── MODEL DEFINITION ──────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class StudentTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers,
                 num_decoder_layers, d_ff, dropout, max_len=512, pad_id=0):
        super().__init__()
        self.d_model = d_model
        self.pad_id  = pad_id
        self.embedding  = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc    = PositionalEncoding(d_model, dropout, max_len)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True,
        )
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.output_proj.weight = self.embedding.weight  # weight tying
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def pad_mask(self, ids):
        return ids == self.pad_id

    def forward(self, src, tgt):
        scale = self.d_model ** 0.5
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=src.device)
        src_emb = self.pos_enc(self.embedding(src) * scale)
        tgt_emb = self.pos_enc(self.embedding(tgt) * scale)
        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=self.pad_mask(src),
            tgt_key_padding_mask=self.pad_mask(tgt),
            memory_key_padding_mask=self.pad_mask(src),
        )
        return self.output_proj(out)


def build_model():
    return StudentTransformer(
        vocab_size=VOCAB_SIZE,
        max_len=MAX_LENGTH + 64,
        pad_id=PAD_ID,
        **model_cfg,
    ).to(device)


model = build_model()
params = sum(p.numel() for p in model.parameters())
print(f"✓ Model Option {MODEL_SIZE}: {params/1e6:.2f}M parameters")

In [ ]:
# ── LOSS, OPTIMIZER, SCHEDULER ────────────────────────────────────────────

class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, pad_id, smoothing=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_id     = pad_id
        self.smoothing  = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits, targets):
        # logits: (N, V)  targets: (N,)
        log_probs = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            smooth = torch.full_like(log_probs, self.smoothing / (self.vocab_size - 2))
            smooth.scatter_(1, targets.unsqueeze(1), self.confidence)
            smooth[:, self.pad_id] = 0.0
        mask = (targets != self.pad_id).float()
        loss = -(smooth * log_probs).sum(dim=-1)
        return (loss * mask).sum() / mask.sum().clamp(min=1)


def get_optimizer(model):
    return torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE,
        betas=(0.9, 0.98), eps=1e-9, weight_decay=WEIGHT_DECAY,
    )


def get_scheduler(optimizer):
    d = model_cfg["d_model"]
    def lr_lambda(step):
        step = max(step, 1)
        return (d ** -0.5) * min(step ** -0.5, step * WARMUP_STEPS ** -1.5)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


criterion = LabelSmoothingLoss(VOCAB_SIZE, PAD_ID, LABEL_SMOOTHING).to(device)
optimizer = get_optimizer(model)
scheduler = get_scheduler(optimizer)
scaler    = GradScaler(enabled=USE_FP16)
print("✓ Loss / Optimizer / Scheduler ready")

In [ ]:
# ── RESUME FROM CHECKPOINT (safe re-run) ──────────────────────────────────
# If a latest checkpoint exists from a previous session, load it and continue.

START_EPOCH     = 1
GLOBAL_STEP     = 0
BEST_DEV_CHRF   = -1.0
NO_IMPROVE      = 0
training_log    = []

if CKPT_LATEST.exists():
    print(f"🔄 Resuming from {CKPT_LATEST.name}...")
    ckpt = torch.load(CKPT_LATEST, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    START_EPOCH   = ckpt["epoch"] + 1
    GLOBAL_STEP   = ckpt["global_step"]
    BEST_DEV_CHRF = ckpt["best_dev_chrf"]
    NO_IMPROVE    = ckpt["no_improve"]
    # Load existing log
    if LOG_CSV.exists():
        training_log = pd.read_csv(LOG_CSV).to_dict("records")
    print(f"  Resumed at epoch {START_EPOCH} | best chrF++={BEST_DEV_CHRF:.2f}")
else:
    print("Starting fresh training run.")

In [ ]:
# ── GREEDY DECODE for per-epoch FLORES BLEU (fast, no beam overhead) ──────
# Full beam search is in 03c. Here we use greedy for speed during training.

@torch.no_grad()
def greedy_translate(model, src_text: str, max_len: int = 100) -> str:
    model.eval()
    src_ids = [BOS_ID] + sp.encode(src_text, add_bos=False, add_eos=False) + [EOS_ID]
    src_ids = src_ids[:MAX_LENGTH]
    src = torch.tensor([src_ids], dtype=torch.long, device=device)

    scale = model.d_model ** 0.5
    src_emb = model.pos_enc(model.embedding(src) * scale)
    memory  = model.transformer.encoder(src_emb, src_key_padding_mask=model.pad_mask(src))

    decoded = [BOS_ID]
    for _ in range(max_len):
        tgt = torch.tensor([decoded], dtype=torch.long, device=device)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(len(decoded), device=device)
        tgt_emb = model.pos_enc(model.embedding(tgt) * scale)
        out  = model.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask)
        next_id = model.output_proj(out[0, -1]).argmax().item()
        if next_id == EOS_ID:
            break
        decoded.append(next_id)

    return sp.decode(decoded[1:])  # strip BOS


def quick_flores_eval(model, src_list, ref_list, n=200):
    """Greedy BLEU on first n FLORES dev sentences. Fast enough for every epoch."""
    hyps = [greedy_translate(model, s) for s in tqdm(src_list[:n], desc="Quick eval", leave=False)]
    bleu = sacrebleu.corpus_bleu(hyps, [ref_list[:n]], tokenize="flores200").score
    chrf = sacrebleu.corpus_chrf(hyps, [ref_list[:n]], word_order=2).score
    return bleu, chrf


print("✓ Greedy decoder ready for quick per-epoch evaluation")

In [ ]:
# ── TRAINING LOOP ─────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, scaler, criterion, step):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(loader, desc="Train", leave=False)

    for i, batch in enumerate(pbar):
        src_b = batch["src"].to(device, non_blocking=True)
        tgt_b = batch["tgt"].to(device, non_blocking=True)
        tgt_in, tgt_out = tgt_b[:, :-1], tgt_b[:, 1:]

        with autocast(enabled=USE_FP16):
            logits = model(src_b, tgt_in)
            loss   = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
            loss   = loss / GRADIENT_ACCUM

        scaler.scale(loss).backward()

        if (i + 1) % GRADIENT_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step += 1

        total_loss += loss.item() * GRADIENT_ACCUM
        pbar.set_postfix(loss=f"{total_loss/(i+1):.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    return total_loss / len(loader), step


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total = 0.0
    for batch in loader:
        src_b = batch["src"].to(device, non_blocking=True)
        tgt_b = batch["tgt"].to(device, non_blocking=True)
        tgt_in, tgt_out = tgt_b[:, :-1], tgt_b[:, 1:]
        with autocast(enabled=USE_FP16):
            logits = model(src_b, tgt_in)
            loss   = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
        total += loss.item()
    return total / len(loader)


def save_ckpt(path, epoch, step, best_chrf, no_improve):
    torch.save({
        "epoch": epoch, "global_step": step,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_dev_chrf": best_chrf,
        "no_improve":    no_improve,
        "model_cfg":     model_cfg,
        "run_name":      RUN_NAME,
    }, path)


print(f"🚀 Training {RUN_NAME}  (epochs {START_EPOCH}–{MAX_EPOCHS})")
print("-" * 60)

for epoch in range(START_EPOCH, MAX_EPOCHS + 1):
    # --- Train ---
    tr_loss, GLOBAL_STEP = train_epoch(
        model, train_loader, optimizer, scheduler, scaler, criterion, GLOBAL_STEP
    )
    # --- Val loss ---
    vl_loss = validate(model, val_loader, criterion)
    # --- Quick FLORES dev (greedy, 200 sentences) ---
    dev_bleu, dev_chrf = quick_flores_eval(model, FLORES_DEV_SRC, FLORES_DEV_REF, n=200)

    # --- Log ---
    row = dict(epoch=epoch, step=GLOBAL_STEP,
               train_loss=round(tr_loss, 4), val_loss=round(vl_loss, 4),
               dev_bleu=round(dev_bleu, 2),  dev_chrf_pp=round(dev_chrf, 2))
    training_log.append(row)
    pd.DataFrame(training_log).to_csv(LOG_CSV, index=False)

    print(f"[{epoch:3d}] train={tr_loss:.4f}  val={vl_loss:.4f}  "
          f"dev_BLEU={dev_bleu:.2f}  dev_chrF++={dev_chrf:.2f}")

    # --- Save latest (for resume) ---
    save_ckpt(CKPT_LATEST, epoch, GLOBAL_STEP, BEST_DEV_CHRF, NO_IMPROVE)

    # --- Save best ---
    if dev_chrf > BEST_DEV_CHRF:
        BEST_DEV_CHRF = dev_chrf
        NO_IMPROVE = 0
        save_ckpt(CKPT_BEST, epoch, GLOBAL_STEP, BEST_DEV_CHRF, 0)
        print(f"  ✅ Best chrF++={BEST_DEV_CHRF:.2f} — saved")
    else:
        NO_IMPROVE += 1
        print(f"  ⏸  No improve ({NO_IMPROVE}/{EARLY_STOP})")
        if NO_IMPROVE >= EARLY_STOP:
            print(f"\n⛔ Early stop at epoch {epoch}")
            break

print(f"\n🏆 Done. Best dev chrF++={BEST_DEV_CHRF:.2f}")
print(f"Best checkpoint: {CKPT_BEST}")
print("→ Now run 03c_evaluate.ipynb for final FLORES devtest scores.")